## 1. Setup

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.paths import MANUAL_PATH, PROCESSED_PATH
from src.risk import herfindahl_index, effective_number_of_positions
from src.exposure import load_bucket_lookthrough, lookthrough_dicts, usd_shares
from src.stress import (
    STRESS_BUCKETS,
    combine_worst_of,
    expand_bucket_scenario,
    fx_translation_shocks,
    historical_shocks,
    resolve_stress_buckets,
    summarize_scenario,
)

## 2. Paths and Config

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

EUR_USD_SHOCK = 0.15              # EUR appreciates 15% vs. USD
SINGLE_POSITION_SHOCK_PCT = -0.50 # largest single stock / crypto position
HISTORICAL_WINDOWS = {
    "Historical: 2022 (calendar year)": ("2022-01-01", "2022-12-31"),
    # Peak close 19 Feb 2020, trough close 23 Mar 2020: the first return
    # inside the drawdown is the one dated 20 Feb.
    "Historical: COVID Crash (Feb-Mar 2020)": ("2020-02-20", "2020-03-23"),
}

## 3. Stress Testing Engine

Helpers live in `src/stress.py` (unit-tested in `tests/test_stress.py`). Scenarios are defined per **stress bucket** (coarse risk category), not per ticker: the tracked code stays free of the actual holdings, and new positions are shocked automatically. The ticker → bucket mapping is the `stress_bucket` column of the gitignored `data/manual/asset_universe_manual.csv`; holdings without one fall back to asset class + region, with a warning. The USD shares behind the EUR/USD scenario come from the currency look-through in the gitignored `data/manual/bucket_lookthrough_manual.csv` (see `src/exposure.py`, shared with notebook 10).

## 4. Load Holdings

In [ ]:
holdings = pd.read_csv(DATA_PATH / "current_holdings_trackable.csv")
holdings = holdings[["yf_ticker", "name", "asset_class", "market_value", "actual_weight", "symbol"]].copy()

universe_path = MANUAL_PATH / "asset_universe_manual.csv"
universe = pd.read_csv(universe_path) if universe_path.exists() else None

ticker_buckets = resolve_stress_buckets(holdings, universe)
holdings["stress_bucket"] = holdings["yf_ticker"].map(ticker_buckets)

# USD share per bucket for the FX scenario — from the currency rows of the
# gitignored bucket look-through file (same source as notebook 10)
lookthrough, _ = lookthrough_dicts(load_bucket_lookthrough(MANUAL_PATH / "bucket_lookthrough_manual.csv"))
bucket_usd_shares = usd_shares(lookthrough)
holdings

## 5. Define Shock Scenarios

Three kinds of scenarios:

- **Hypothetical** — instantaneous shocks per stress bucket (judgement-based what-ifs).
- **Historical** — each holding's actual compounded EUR return over a past window, from `asset_returns_eur.csv` (history-proxy back-filled, see Notebook 03). No hand-typed numbers.
- **FX** — pure currency translation of a stronger EUR: USD-denominated value falls by 1/(1 + move) − 1 (−13.0% for EUR +15%, not −15%), scaled by each bucket's USD share (look-through for funds; gold and crypto are USD-priced).

**Limitations** — all shocks are instantaneous and applied to today's holdings: no path over time, no correlations beyond what a scenario encodes, no rebalancing during the stress. Hypothetical magnitudes and the USD shares per bucket are judgement calls, not estimates. Historical scenarios use a holding's history proxy (same index) before its own listing date. Read the results as orders of magnitude, not forecasts.

In [ ]:
BUCKET_SCENARIOS = {
    "Global Equity Crash": {
        "global_equity": -0.30, "em_equity": -0.30, "europe_equity": -0.30, "small_cap": -0.30,
        "tech_thematic": -0.35, "us_single_stock": -0.32, "eu_single_stock": -0.30,
        "gold": 0.05, "crypto": -0.35,
    },
    "Tech Crash": {
        "tech_thematic": -0.45, "us_single_stock": -0.45, "eu_single_stock": -0.10,
        "global_equity": -0.15, "em_equity": -0.10, "europe_equity": -0.10, "small_cap": -0.15,
        "gold": 0.0, "crypto": -0.30,
    },
    "Crypto Crash": {
        "crypto": -0.60,
    },
    "Interest Rate Shock": {
        "tech_thematic": -0.20, "us_single_stock": -0.17, "eu_single_stock": -0.08,
        "global_equity": -0.10, "em_equity": -0.08, "europe_equity": -0.08, "small_cap": -0.10,
        "gold": -0.08, "crypto": -0.15,
    },
    "US Political / Policy Shock": {
        "global_equity": -0.20, "small_cap": -0.22, "tech_thematic": -0.25, "us_single_stock": -0.22,
        "em_equity": -0.12, "europe_equity": -0.08, "eu_single_stock": -0.08,
        "gold": 0.08, "crypto": -0.15,
    },
}

missing_buckets = {
    name: sorted(set(STRESS_BUCKETS) - set(shocks))
    for name, shocks in BUCKET_SCENARIOS.items()
    if name != "Crypto Crash" and set(STRESS_BUCKETS) - set(shocks)
}
assert not missing_buckets, f"Scenarios without a shock for some buckets: {missing_buckets}"

SHOCK_SCENARIOS = {
    name: expand_bucket_scenario(shocks, ticker_buckets)
    for name, shocks in BUCKET_SCENARIOS.items()
}

asset_returns = pd.read_csv(DATA_PATH / "asset_returns_eur.csv", index_col=0, parse_dates=True)
for name, (start, end) in HISTORICAL_WINDOWS.items():
    SHOCK_SCENARIOS[name] = historical_shocks(asset_returns, start, end)

SHOCK_SCENARIOS[f"EUR/USD Shock (EUR +{EUR_USD_SHOCK:.0%})"] = fx_translation_shocks(EUR_USD_SHOCK, ticker_buckets, bucket_usd_shares)

pd.DataFrame(SHOCK_SCENARIOS).round(3)

## 5b. Large Single-Position Loss & Combined Stress

Concentration risk means one *company* failing — so the single-position scenario hits the largest direct stock or crypto position, not a diversified fund.

In [ ]:
single_positions = holdings[holdings["asset_class"].isin(["STOCK", "CRYPTO"])]

if not single_positions.empty:
    largest_single = single_positions.loc[single_positions["market_value"].idxmax()]
    SHOCK_SCENARIOS[f"Large Single-Position Loss ({largest_single['name']})"] = {
        largest_single["yf_ticker"]: SINGLE_POSITION_SHOCK_PCT
    }

SHOCK_SCENARIOS["Combined Stress (Equity Crash + Rate Shock)"] = combine_worst_of(
    SHOCK_SCENARIOS["Global Equity Crash"], SHOCK_SCENARIOS["Interest Rate Shock"]
)

## 6. Apply All Scenarios

In [ ]:
scenario_results = {}
scenario_detail = {}

for scenario_name, shocks in SHOCK_SCENARIOS.items():
    summary, detail = summarize_scenario(holdings, shocks)
    scenario_results[scenario_name] = summary
    scenario_detail[scenario_name] = detail

scenario_summary_df = pd.DataFrame(scenario_results).T
scenario_summary_df.sort_values("total_loss_pct")

## 7. Loss Contribution by Asset

In [ ]:
loss_by_asset_all = []
for scenario_name, detail in scenario_detail.items():
    df = detail[[
        "yf_ticker", "name", "asset_class", "market_value",
        "loss_eur", "loss_pct_of_position", "new_weight"
    ]].copy()
    df.insert(0, "scenario", scenario_name)
    loss_by_asset_all.append(df)

loss_by_asset_all = pd.concat(loss_by_asset_all, ignore_index=True)

FOCUS_SCENARIO = "Global Equity Crash"
loss_by_asset_all[loss_by_asset_all["scenario"] == FOCUS_SCENARIO].sort_values("loss_eur")

## 8. Loss Contribution by Asset Class

In [ ]:
loss_by_class_all = (
    loss_by_asset_all
    .groupby(["scenario", "asset_class"])[["market_value", "loss_eur"]]
    .sum()
    .reset_index()
)
loss_by_class_all["loss_pct_of_class"] = loss_by_class_all["loss_eur"] / loss_by_class_all["market_value"]
loss_by_class_all[loss_by_class_all["scenario"] == FOCUS_SCENARIO]

## 9. Portfolio Composition Before vs. After Shock

In [ ]:
composition_compare_all = loss_by_asset_all.copy()
composition_compare_all["weight_before"] = composition_compare_all["yf_ticker"].map(
    holdings.set_index("yf_ticker")["actual_weight"]
)
composition_compare_all["weight_change_pp"] = (
    (composition_compare_all["new_weight"] - composition_compare_all["weight_before"]) * 100
)
composition_compare_all[composition_compare_all["scenario"] == FOCUS_SCENARIO].sort_values("weight_change_pp")

## 10. Risk After Shock

In [ ]:
hhi_before = herfindahl_index(holdings["actual_weight"])
eff_n_before = effective_number_of_positions(holdings["actual_weight"])

risk_after_shock_rows = []
for scenario_name, detail in scenario_detail.items():
    risk_after_shock_rows.append({
        "scenario": scenario_name,
        "hhi_before": hhi_before,
        "hhi_after": herfindahl_index(detail["new_weight"]),
        "effective_n_before": eff_n_before,
        "effective_n_after": effective_number_of_positions(detail["new_weight"]),
    })

risk_after_shock_df = pd.DataFrame(risk_after_shock_rows).set_index("scenario")
risk_after_shock_df

## 11. Visualization

In [ ]:
plot_df = scenario_summary_df.sort_values("total_loss_pct")

plt.figure(figsize=(10, 6))
plt.barh(plot_df.index, plot_df["total_loss_pct"] * 100)
plt.xlabel("Portfolio Loss (%)")
plt.title("Stress Test: Portfolio Loss by Scenario")
plt.tight_layout()
plt.show()

In [ ]:
focus_detail = loss_by_asset_all[loss_by_asset_all["scenario"] == FOCUS_SCENARIO].sort_values("loss_eur")

plt.figure(figsize=(10, 6))
plt.barh(focus_detail["name"], focus_detail["loss_eur"])
plt.xlabel("Loss (EUR)")
plt.title(f"Loss Contribution by Asset — {FOCUS_SCENARIO}")
plt.tight_layout()
plt.show()

## 12. Rebalancing Reaction

In [ ]:
target_weights_raw = pd.read_csv(DATA_PATH / "actual_vs_target_weights.csv")
target_weights = (
    target_weights_raw
    .merge(holdings[["symbol", "yf_ticker"]], on="symbol", how="left")
    .dropna(subset=["yf_ticker"])
    .set_index("yf_ticker")["target_weight"]
    .astype(float)
)

worst_scenario = scenario_summary_df["total_loss_pct"].idxmin()

post_shock_vs_target = pd.DataFrame({
    "New Weight (Post-Shock)": scenario_detail[worst_scenario].set_index("yf_ticker")["new_weight"],
    "Target Weight": target_weights,
})
post_shock_vs_target["Drift (pp)"] = (
    (post_shock_vs_target["New Weight (Post-Shock)"] - post_shock_vs_target["Target Weight"]) * 100
)
post_shock_vs_target.sort_values("Drift (pp)")

## 13. Export

In [ ]:
scenario_summary_df.to_csv(DATA_PATH / "stress_test_scenario_summary.csv")
loss_by_asset_all.to_csv(DATA_PATH / "stress_test_loss_by_asset.csv", index=False)
loss_by_class_all.to_csv(DATA_PATH / "stress_test_loss_by_class.csv", index=False)
composition_compare_all.to_csv(DATA_PATH / "stress_test_composition_change.csv", index=False)
risk_after_shock_df.to_csv(DATA_PATH / "stress_test_risk_after_shock.csv")

print("All stress test exports saved.")